# DEM API Validation

Validates that the TNM API approach returns the same data as the manually downloaded tiles.

**Strategy:** Pick a small bounding box inside a tile we already have (`USGS_1M_12_x68y364_NM_SouthCentral_2018_D19.tif`), fetch it programmatically via the TNM API, clip both to the same extent, and compare pixel-by-pixel.

In [ ]:
import requests
import numpy as np
import rasterio
import rasterio.mask
from rasterio.transform import from_bounds
from rasterio.warp import reproject, Resampling
from pyproj import Transformer
from shapely.geometry import box
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pathlib import Path
import tempfile
import shutil

DEM_DIR = Path("../data/raw/dem")
REFERENCE_TILE = DEM_DIR / "USGS_1M_12_x68y364_NM_SouthCentral_2018_D19.tif"

## 1. Define test AOI

Small 0.05° × 0.05° box entirely inside the reference tile (lat 32.793–32.882, lon –109.078––108.969).

In [ ]:
# small chunk near the center of the reference tile
TEST_BBOX = (-109.050, 32.820, 32.820, 32.860)  # minLon, minLat, maxLon, maxLat
TEST_BBOX = (-109.050, 32.820, -109.000, 32.860)

minlon, minlat, maxlon, maxlat = TEST_BBOX
print(f"Test AOI: lon {minlon} to {maxlon}, lat {minlat} to {maxlat}")
print(f"Size: ~{(maxlon - minlon) * 111:.1f} km W-E, ~{(maxlat - minlat) * 111:.1f} km N-S")

## 2. Query TNM API for this AOI

In [ ]:
def query_tnm(bbox, resolution="1 meter", max_items=100):
    """Query USGS TNM API and return list of (filename, downloadURL)."""
    minlon, minlat, maxlon, maxlat = bbox
    params = {
        "datasets": f"Digital Elevation Model (DEM) {resolution}",
        "bbox": f"{minlon},{minlat},{maxlon},{maxlat}",
        "prodFormats": "GeoTIFF",
        "max": max_items,
        "outputFormat": "JSON",
    }
    r = requests.get(
        "https://tnmaccess.nationalmap.gov/api/v1/products",
        params=params,
        timeout=30,
    )
    r.raise_for_status()
    data = r.json()
    total = data.get("total", 0)
    items = data.get("items", [])
    print(f"API returned {len(items)} of {total} total products")
    if total > max_items:
        print(f"WARNING: {total - max_items} products were truncated — increase max_items")
    return [(i["title"], i["downloadURL"]) for i in items]

products = query_tnm(TEST_BBOX)
for title, url in products:
    print(f"  {title}")
    print(f"    {url}")

## 3. Check whether the API URL matches our manually downloaded tile

In [ ]:
api_filenames = [Path(url).name for _, url in products]
manual_filename = REFERENCE_TILE.name

print(f"Manual tile   : {manual_filename}")
print(f"API tile(s)   : {api_filenames}")
print()

match = manual_filename in api_filenames
print(f"Filename match: {'YES ✓' if match else 'NO — different tile(s) returned'}")
if not match:
    print("Note: AOI may straddle a tile boundary — API could return adjacent tile(s)")

## 4. Download the API tile to a temp directory

In [ ]:
def download_file(url, dest_path, chunk_size=1 << 20):
    with requests.get(url, stream=True, timeout=120) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0))
        downloaded = 0
        with open(dest_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=chunk_size):
                f.write(chunk)
                downloaded += len(chunk)
        print(f"Downloaded {downloaded / 1e6:.1f} MB → {dest_path.name}")

tmp_dir = Path(tempfile.mkdtemp(prefix="dem_api_test_"))
print(f"Temp dir: {tmp_dir}")

api_paths = []
for title, url in products:
    dest = tmp_dir / Path(url).name
    if not dest.exists():
        download_file(url, dest)
    else:
        print(f"Already exists: {dest.name}")
    api_paths.append(dest)

## 5. Clip both rasters to the test AOI extent

In [ ]:
def clip_to_bbox(raster_path, bbox_lonlat):
    """Clip raster to bbox (in lon/lat) and return (array, transform, crs, nodata)."""
    minlon, minlat, maxlon, maxlat = bbox_lonlat
    with rasterio.open(raster_path) as src:
        # convert bbox to raster CRS
        t = Transformer.from_crs("EPSG:4326", src.crs, always_xy=True)
        xmin, ymin = t.transform(minlon, minlat)
        xmax, ymax = t.transform(maxlon, maxlat)
        geom = box(xmin, ymin, xmax, ymax)
        out_image, out_transform = rasterio.mask.mask(
            src, [geom.__geo_interface__], crop=True, nodata=src.nodata
        )
        return out_image[0].astype(np.float32), out_transform, src.crs, src.nodata

manual_arr, manual_tf, manual_crs, manual_nodata = clip_to_bbox(REFERENCE_TILE, TEST_BBOX)
print(f"Manual clip shape : {manual_arr.shape}")
print(f"Manual value range: {np.nanmin(manual_arr):.2f} – {np.nanmax(manual_arr):.2f} m")

# if API returned multiple tiles, use the matching one; otherwise use first
api_tile = next((p for p in api_paths if p.name == manual_filename), api_paths[0])
api_arr, api_tf, api_crs, api_nodata = clip_to_bbox(api_tile, TEST_BBOX)
print(f"\nAPI clip shape    : {api_arr.shape}")
print(f"API value range   : {np.nanmin(api_arr):.2f} – {np.nanmax(api_arr):.2f} m")

## 6. Align grids (snap to same pixel grid if needed)

In [ ]:
def align_to_reference(src_arr, src_tf, src_crs, ref_arr, ref_tf, ref_crs):
    """Reproject src onto ref pixel grid. Returns aligned array."""
    if src_arr.shape == ref_arr.shape and src_tf == ref_tf:
        print("Grids already aligned — no reprojection needed")
        return src_arr
    print(f"Grid mismatch: {src_arr.shape} vs {ref_arr.shape} — reprojecting...")
    aligned = np.full_like(ref_arr, np.nan)
    reproject(
        source=src_arr,
        destination=aligned,
        src_transform=src_tf,
        src_crs=src_crs,
        dst_transform=ref_tf,
        dst_crs=ref_crs,
        resampling=Resampling.nearest,
    )
    return aligned

api_aligned = align_to_reference(api_arr, api_tf, api_crs, manual_arr, manual_tf, manual_crs)
print(f"Final shapes — manual: {manual_arr.shape}, api: {api_aligned.shape}")

## 7. Statistical comparison

In [ ]:
# mask nodata
nodata_val = manual_nodata if manual_nodata is not None else -9999
valid = (manual_arr != nodata_val) & (api_aligned != nodata_val) & \
        np.isfinite(manual_arr) & np.isfinite(api_aligned)

diff = manual_arr[valid] - api_aligned[valid]

print("=== Pixel-level difference stats (manual − API) ===")
print(f"  Valid pixels  : {valid.sum():,} of {valid.size:,}")
print(f"  Min diff      : {diff.min():.6f} m")
print(f"  Max diff      : {diff.max():.6f} m")
print(f"  Mean diff     : {diff.mean():.6f} m")
print(f"  RMSE          : {np.sqrt((diff**2).mean()):.6f} m")
print(f"  Identical px  : {(diff == 0).sum():,} ({100*(diff==0).mean():.1f}%)")

## 8. Visual comparison

In [ ]:
diff_img = np.full_like(manual_arr, np.nan)
diff_img[valid] = diff

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# manual DEM
im0 = axes[0].imshow(manual_arr, cmap="terrain")
axes[0].set_title("Manual download\n(reference tile clipped)")
plt.colorbar(im0, ax=axes[0], label="Elevation (m)", shrink=0.8)

# API DEM
im1 = axes[1].imshow(api_aligned, cmap="terrain")
axes[1].set_title("API download\n(TNM programmatic)")
plt.colorbar(im1, ax=axes[1], label="Elevation (m)", shrink=0.8)

# difference
absmax = np.nanmax(np.abs(diff_img)) if np.any(np.isfinite(diff_img)) else 1
norm = mcolors.TwoSlopeNorm(vmin=-absmax, vcenter=0, vmax=absmax)
im2 = axes[2].imshow(diff_img, cmap="RdBu", norm=norm)
axes[2].set_title("Difference (manual − API)")
plt.colorbar(im2, ax=axes[2], label="Δ elevation (m)", shrink=0.8)

for ax in axes:
    ax.axis("off")

plt.suptitle(
    f"Test AOI: lon {TEST_BBOX[0]}–{TEST_BBOX[2]}, lat {TEST_BBOX[1]}–{TEST_BBOX[3]}",
    fontsize=12,
)
plt.tight_layout()
plt.savefig("../outputs/dem_api_validation.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved to outputs/dem_api_validation.png")

## 9. Histogram of differences

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(diff, bins=50, color="steelblue", edgecolor="white", linewidth=0.3)
ax.axvline(0, color="red", linewidth=1.5, linestyle="--", label="zero diff")
ax.set_xlabel("Elevation difference (m)")
ax.set_ylabel("Pixel count")
ax.set_title("Distribution of (manual − API) differences")
ax.legend()
plt.tight_layout()
plt.show()

## 10. Verdict

In [ ]:
rmse = np.sqrt((diff**2).mean())
pct_identical = 100 * (diff == 0).mean()

if rmse < 0.01 and pct_identical > 99:
    verdict = "PASS — API and manual tiles are effectively identical. Safe to use TNM API for all future downloads."
elif rmse < 0.1:
    verdict = f"PASS with minor float differences (RMSE={rmse:.4f}m). Likely floating-point precision only. API approach is valid."
else:
    verdict = f"FAIL — RMSE={rmse:.4f}m. Investigate: different tile versions, resampling, or wrong tile matched."

print(verdict)

## Cleanup

In [ ]:
# uncomment to delete the temp downloaded file
# shutil.rmtree(tmp_dir)
# print(f"Deleted {tmp_dir}")
print(f"Temp files kept at: {tmp_dir}")